In [1]:
from simsopt.mhd import Boozer,Vmec
from simsopt.geo import Surface,SurfaceRZFourier
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import neo
from neo import NeoContext,neo_surfaces_from_simsopt_boozer
import time
import numpy as np
from pathlib import Path


mgrid_candidates = [
    Path("tests/test_file/mgrid_c09r00.nc"),
    Path("test_file/mgrid_c09r00.nc"),
]
mgrid_path = next((p for p in mgrid_candidates if p.exists()), None)
if mgrid_path is None:
    raise FileNotFoundError(
        "Cannot find mgrid file. Tried: " + ", ".join(str(p) for p in mgrid_candidates)
    )


# initial_rz = (1.57,0)
# mgrid_filename = str(mgrid_path)
# extcur = None
# vmec_path = "/home/zkg/ripplepy/tests/test_file/wout_ncsx_c09r00_free.nc"
initial_rz = (1.26,0)
mgrid_filename = '/Users/zkgao/ripplepy/tests/test_file/mgrid_h1_design.nc'
extcur = [50000, 5000, 1, -80000, -40000]
vmec_path = "/Users/zkgao/ripplepy/tests/test_file/wout_h1_design.nc"

sur_idx = np.linspace(0.1, 0.5, 10)

RZ_points = []  # 每个元素是 [R, phi, Z]
for s in sur_idx:
    surface = SurfaceRZFourier.from_wout(vmec_path, s)
    RphiZ = surface.cross_section(phi=0)[0]
    RZ = RphiZ[[0, 2]]   # shape: (Npts, 3)
    RZ_points.append(RZ)             # 第一个点: (3,)

RZ_points = np.asarray(RZ_points)       # shape: (len(sur_idx), 3) [R1,Z1], ...]

vmec = Vmec(str(vmec_path))
boozer = Boozer(vmec)
boozer.mpol = 24
boozer.ntor = 72
# ns_list =  np.array([2, 3, 4, 5, 6, 7, 8, 9, 10])
# boozer.register(ns_list/100)
boozer.register(np.linspace(0.1, 1.0, 10))
boozer.bx.verbose =True
boozer.run()
print("Boozer coefficients computed.")
# boozer.bx.write_boozmn("test_file/boozmn_h1_design.nc")

Read ns=300, mpol=Boozer coefficients computed.
14, ntor=14, mnmax=392, mnmax_nyq=578
compute_surfs (0-based indices):  29 59 89 119 149 179 209 239 269 298
Initializing with mboz=24, nboz=72
ntheta = 98, nzeta = 290, # threads = 0
                   |        outboard (theta=0)      |      inboard (theta=pi)      |
thread js_b js zeta| |B|input  |B|Boozer    Error   | |B|input  |B|Boozer    Error |
------------------------------------------------------------------------------------
   0     0  29   0  3.376e-01  3.376e-01  6.225e-07  3.128e-01  3.128e-01  2.121e-06
                pi  4.789e-01  4.789e-01  4.963e-06  4.926e-01  4.926e-01  2.687e-06
   0     1  59   0  3.434e-01  3.434e-01  9.761e-06  3.081e-01  3.081e-01  4.013e-07
                pi  4.768e-01  4.768e-01  1.566e-05  4.964e-01  4.964e-01  4.832e-06
   0     2  89   0  3.480e-01  3.480e-01  6.395e-06  3.046e-01  3.047e-01  2.738e-05
                pi  4.753e-01  4.753e-01  2.776e-05  4.995e-01  4.994e-01  1.548e-05
   

In [ ]:
neoclass = neo.from_simsopt_boozer(boozer)
ctx = NeoContext()
ctx.set_boozer(neoclass)
surfaces = neo_surfaces_from_simsopt_boozer(boozer)
print('Surfaces from simsopt Boozer:', surfaces)
ctx.set_flux_surfaces(surfaces.tolist())
ctx.set_resolution(theta_n=200, phi_n=200)
ctx.set_mode_limits(max_m_mode=0, max_n_mode=0)
ctx.set_transport_options(
    npart=100,

    multra=1,
    acc_req=0.01,
    no_bins=100,
    nstep_per=50,
    nstep_min=500,
    nstep_max=5000,
    calc_nstep_max=0,
)
ctx.set_switches(ref_swi=2, eout_swi=2, calc_cur=0)
ctx.set_output_options(
    write_progress=0,
    write_output_files=0,
    write_integrate=0,
    write_diagnostic=0,
    suppress_file_io=True,
)

ctx.setup_grids()
ctx.run_all()

got_surfaces = ctx.surface_map()
got_epstot = ctx.epstot_profile()

In [ ]:
import numpy as np
from ripplepy import MGrid

# from ripplepy.effective_ripple import Effective_Ripple
from ripplepy import set_extcur, initialize_mgrid_field,set_trace_parameters,compute_epstot,find_axis
import time
from func_timeout import func_timeout, FunctionTimedOut
from pathlib import Path

full_torus = False
nfp = 3

mgrid_candidates = [
    Path("tests/test_file/mgrid_c09r00.nc"),
    Path("test_file/mgrid_c09r00.nc"),
]
mgrid_path = next((p for p in mgrid_candidates if p.exists()), None)
if mgrid_path is None:
    raise FileNotFoundError(
        "Cannot find mgrid file. Tried: " + ", ".join(str(p) for p in mgrid_candidates)
    )

mgrid_filename = str(mgrid_path)
# extcur = [6.52271941985300E+05, 6.51868569367400E+05, 5.37743588647300E+05, 2.50000000000000E-07, 2.50000000000000E-07, 2.80949750000000E+04, -5.48049500000000E+04, 3.01228950000000E+04, 9.42409100000000E+04, 4.55138737653200E+04]
# extcur = np.ones(10)
extcur = None


# mgrid_filename = '/home/zkg/CN_H1_scan_fieldlines/H1_design/coils/mgrid_h1_design.nc'
# extcur = [50000, 5000, 1, -80000, -40000]
# initial_rz = (1.26,0)

nturn = 200
nphi = 360
initialize_mgrid_field(mgrid_filename, nfp,full_torus=full_torus)

extcur=set_extcur(extcur)
# sum_bfield_internal = True

axis_rz, R0, axis_fieldline, trace_istate = find_axis(initial_rz, xtol=1e-5, max_iter=100)
print(f"✓ Magnetic axis found at R={axis_rz[0]:.10f}, Z={axis_rz[1]:.10f}, R0={R0:.10f}")

# initial_gradpsi = compute_initial_gradpsi(extcur,initial_rz[0], initial_rz[1], phi0=0.0,verbose=False)
initial_gradpsi = [1,0,0]
initial_gradpsi = np.array(initial_gradpsi, dtype=np.float64, order='F')
set_trace_parameters(nturn, nphi)
ripplepy_results = []
for i in RZ_points:

    fieldline_data = np.zeros((nturn*nphi, 20), dtype=np.float64, order='F')
    geocur = np.zeros((nturn*nphi, 3), dtype=np.float64, order='F')
    R0 = np.array(R0, dtype=np.float64, order='F')
    Bboundary = np.array(0.0, dtype=np.float64, order='F')
    initial_rz = np.array(i, dtype=np.float64, order='F')
    # # 预分配轨线数据缓冲区并传入 Fortran 例程（Fortran 连续）

    starttime = time.time()
    epsilon_eff, Bboundary,trace_istate = compute_epstot(R0,extcur, initial_rz, initial_gradpsi, fieldline_data)
    endtime = time.time()
    time_elapsed = endtime - starttime
    print(f"✓ Ripple fieldline computed @ {initial_rz[0]:.10f}, {initial_rz[1]:.10f}. epsilon_eff={epsilon_eff:.6e}, Bboundary={Bboundary:.6e},time={time_elapsed:.3f} s")
    ripplepy_results.append(( epsilon_eff))

print("ripplepy_results:", ripplepy_results)
ripplepy_200nturn_results = ripplepy_results

nturn = 400
set_trace_parameters(nturn, nphi)
ripplepy_results = []
for i in RZ_points:

    fieldline_data = np.zeros((nturn*nphi, 20), dtype=np.float64, order='F')
    geocur = np.zeros((nturn*nphi, 3), dtype=np.float64, order='F')
    R0 = np.array(R0, dtype=np.float64, order='F')
    Bboundary = np.array(0.0, dtype=np.float64, order='F')
    initial_rz = np.array(i, dtype=np.float64, order='F')
    # # 预分配轨线数据缓冲区并传入 Fortran 例程（Fortran 连续）

    starttime = time.time()
    epsilon_eff, Bboundary, trace_istate = compute_epstot(R0,extcur, initial_rz, initial_gradpsi, fieldline_data)
    endtime = time.time()
    time_elapsed = endtime - starttime
    print(f"✓ Ripple fieldline computed @ {initial_rz[0]:.10f}, {initial_rz[1]:.10f}. epsilon_eff={epsilon_eff:.6e}, Bboundary={Bboundary:.6e},time={time_elapsed:.3f} s")
    ripplepy_results.append(( epsilon_eff))

print("ripplepy_results:", ripplepy_results)
ripplepy_400nturn_results = ripplepy_results

In [ ]:
ref_surfaces = RZ_points[:,0]  # 假设 RZ_points 的第一列是表面索引或半径
ripplepy_200nturn_results = np.array(ripplepy_200nturn_results)
ripplepy_400nturn_results = np.array(ripplepy_400nturn_results)

print('pyneo:', got_epstot)
print('ripplepy_200nturn:', ripplepy_200nturn_results)
print('ripplepy_400nturn:', ripplepy_400nturn_results)
print('ref_surfaces:', ref_surfaces)

plt.figure()
plt.plot(ref_surfaces, ripplepy_200nturn_results, label="ripplepy_200nturn")
plt.plot(ref_surfaces, ripplepy_400nturn_results, label="ripplepy_400nturn")
plt.plot(ref_surfaces, got_epstot, label="pyneo")
plt.xlabel("R")
plt.ylabel("Epsilon")
plt.legend()
plt.show()

# plot percentage error relative to pyneo
plt.figure()
percent_error = np.where(got_epstot != 0, (ripplepy_200nturn_results - got_epstot) / got_epstot * 100.0, np.nan)
plt.errorbar(ref_surfaces, percent_error, yerr=np.abs(percent_error) * 0.1, fmt='o', label="relative percent error vs pyneo")
plt.xlabel("R")
plt.ylabel("Percent error (%)")
plt.legend()